# 🛡️ HallucinationGuard v2 — Linear Probe Proof on Qwen3.6-27B

**What changed vs notebook 30**: notebook 30 used a *single SAE feature* (f34957 at L31) as score. AUROC on Ferrando entity test was 0.81, but on public hallucination benchmarks it collapsed to ~0.50 (chance).

**Hypothesis (notebook 31)**: a *linear probe trained on raw residual stream* generalizes better. Notebook 28 reported LR L32 AUROC = 0.887 on the same narrow Ferrando task — let's see if multi-dim probe transfers to TruthfulQA + HaluEval + SimpleQA + MMLU.

## The 3 questions, again

| Question | Metric | Pass threshold |
|---|---|---|
| **Detection works?** | LR probe AUROC, held-out per benchmark | ≥ 0.70 each |
| **Generalizes?** | LR trained on 3 benches, evaluated on 4th | ≥ 0.65 cross-bench |
| **Mitigation works?** | Confident-wrong reduction with abstain mode | ≥ 30%, MMLU loss ≤ 3pp |

## Strategy

1. Re-run all 4 benchmarks with **residual capture** at L31 + L32 (Ferrando-style: last token of question prompt)
2. Build feature matrix `X` (N × d_model) and labels `y` (1 = hallucinated, 0 = correct)
3. Train LogReg with L2 (C swept on 5-fold CV) **per benchmark** (within-bench AUROC)
4. Train LogReg **across benchmarks**, evaluate held-out per benchmark (cross-bench AUROC = generalization test)
5. Compare: SAE feature single (notebook 30) vs LR within-bench vs LR cross-bench
6. If cross-bench AUROC ≥ 0.65 across all: **🟢 GREEN LIGHT** for HallucinationGuard universal
7. If only within-bench works: ship as **EntityRecognitionGuard** + per-domain probes (smaller TAM)
8. If neither works: pivot deeper

## Time + cost

RTX 6000 96 GB Colab. ~5 min model load (cached) + ~30 min benchmark residual capture + ~5 min probe train + ~10 min mitigation rerun = **~50 min**, ~R$10.

In [ ]:
!pip -q install --upgrade transformers accelerate safetensors huggingface_hub datasets scipy scikit-learn matplotlib tqdm

## 1. Setup + auth + load Qwen3.6-27B

Same as notebook 30. If you've just finished notebook 30, model is already loaded — you can skip this section and reuse `model`, `tok`, `blocks` from the previous notebook session.

**Self-contained version below** (works from cold start).

In [ ]:
import os, json, time, math, gc
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from huggingface_hub import login, hf_hub_download, HfApi, create_repo

HF_TOKEN = os.environ.get('HF_TOKEN')
if HF_TOKEN is None:
    import getpass
    HF_TOKEN = getpass.getpass('HF token (write scope): ')
login(HF_TOKEN, add_to_git_credential=False)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda', 'Need GPU.'
print(f'CUDA: {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

CFG = {
    'model':            'Qwen/Qwen3.6-27B',
    'probe_layers':     [31, 32],            # collect at both, train at each, pick best
    'subset_size': {
        'truthfulqa': 200,
        'haluval':    200,
        'simpleqa':   100,
        'mmlu':       500,
    },
    'train_test_split': 0.8,
    'random_seed':      42,
    'lr_C_sweep':       [0.001, 0.01, 0.1, 1.0, 10.0],
    'guard_threshold_init': 0.5,
    'max_new_tokens':   128,
    'hf_results_repo':  os.environ.get('HF_USERNAME', 'caiovicentino1') + '/hallucinationguard-v2-linearprobe-qwen36-27b',
}
LOCAL_OUT = Path('/content/hg_v2_out')
LOCAL_OUT.mkdir(parents=True, exist_ok=True)
print(json.dumps(CFG, indent=2, default=str))

In [ ]:
# Skip this cell if model is already loaded from notebook 30
from transformers import AutoTokenizer, AutoModelForImageTextToText, AutoModelForCausalLM

if 'model' not in globals() or 'tok' not in globals():
    print(f'Loading {CFG["model"]} ...')
    tok = AutoTokenizer.from_pretrained(CFG['model'], trust_remote_code=True)
    try:
        model = AutoModelForImageTextToText.from_pretrained(
            CFG['model'], dtype=torch.bfloat16, attn_implementation='sdpa',
            device_map={'': device}, trust_remote_code=True,
        )
    except Exception:
        model = AutoModelForCausalLM.from_pretrained(
            CFG['model'], dtype=torch.bfloat16, attn_implementation='sdpa',
            device_map={'': device}, trust_remote_code=True,
        )
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)

    def _block_list(m):
        candidates = [m]
        if hasattr(m, 'model'):
            candidates.append(m.model)
        for s in candidates:
            for path in [('model','language_model','layers'), ('language_model','layers'),
                         ('model','layers'), ('layers',)]:
                cur = s; ok = True
                for p in path:
                    if hasattr(cur, p): cur = getattr(cur, p)
                    else: ok = False; break
                if ok and hasattr(cur, '__getitem__'):
                    return cur
        raise RuntimeError('layers not found')
    blocks = _block_list(model)
else:
    print('Reusing model/tok/blocks from notebook 30 session.')

d_model = (model.config.text_config.hidden_size if hasattr(model.config, 'text_config')
           else model.config.hidden_size)
print(f'Model: {len(blocks)} layers, d_model = {d_model}')
print(f'GPU mem: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## 2. Multi-layer residual capture

Hook each target layer (L31, L32). For each prompt, capture the **last-token residual** at each layer simultaneously. One forward pass yields all layer residuals.

In [ ]:
class MultiLayerHook:
    """Capture residual at multiple layers in one forward pass."""
    def __init__(self, blocks, layers):
        self.layers = layers
        self.bufs = {l: None for l in layers}
        self.handles = []
        for l in layers:
            self.handles.append(blocks[l].register_forward_hook(self._make(l)))
    def _make(self, l):
        def hook(_mod, _inp, out):
            h = out[0] if isinstance(out, tuple) else out
            self.bufs[l] = h.detach()
        return hook
    def pop(self, layer):
        b = self.bufs[layer]; self.bufs[layer] = None; return b
    def close(self):
        for h in self.handles:
            h.remove()

ml_hook = MultiLayerHook(blocks, CFG['probe_layers'])

@torch.no_grad()
def capture_last_token_residuals(prompt, max_input_length=512):
    """Forward + capture last-token residual at each probe layer."""
    enc = tok(prompt, return_tensors='pt', truncation=True, max_length=max_input_length).to(device)
    n_valid = enc['attention_mask'].sum().item()
    last_pos = n_valid - 1
    model(**enc)
    out = {}
    for l in CFG['probe_layers']:
        h = ml_hook.pop(l)                           # (1, T, D) bf16
        out[l] = h[0, last_pos].float().cpu().numpy()  # (D,)
    return out

# Quick test
r = capture_last_token_residuals('Q: What is the capital of France?\nA:')
for l, h in r.items():
    print(f'  L{l}: shape {h.shape}, norm {np.linalg.norm(h):.2f}')

## 3. Re-run benchmarks with residual capture

Same logic as notebook 30 (chat template + log-prob picker for multi-choice, generation + substring match for QA), but adds residual capture per question. Stores `(residuals, label, prompt)` for probe training.

In [ ]:
from datasets import load_dataset
from tqdm.auto import tqdm
import random
random.seed(CFG['random_seed'])
np.random.seed(CFG['random_seed'])

# Load benchmarks
def _safe_load(loader_fn, name):
    try:
        return loader_fn()
    except Exception as e:
        print(f'  [{name}] failed: {e}'); return None

tqa = _safe_load(lambda: load_dataset('truthful_qa', 'multiple_choice', split='validation'), 'truthfulqa')
halu = _safe_load(lambda: load_dataset('pminervini/HaluEval', 'qa', split='data'), 'haluval')
if halu is None:
    halu = _safe_load(lambda: load_dataset('Sneha/HaluEval', 'qa', split='train'), 'haluval-fb')
simpleqa = _safe_load(lambda: load_dataset('basicv8vc/SimpleQA', split='test'), 'simpleqa')
mmlu = _safe_load(lambda: load_dataset('cais/mmlu', 'all', split='test'), 'mmlu')

# Sample subsets
subsets = {}
if tqa is not None:
    n = min(CFG['subset_size']['truthfulqa'], len(tqa))
    subsets['truthfulqa'] = [tqa[i] for i in random.sample(range(len(tqa)), n)]
if halu is not None:
    n = min(CFG['subset_size']['haluval'], len(halu))
    subsets['haluval'] = [halu[i] for i in random.sample(range(len(halu)), n)]
if simpleqa is not None:
    n = min(CFG['subset_size']['simpleqa'], len(simpleqa))
    subsets['simpleqa'] = [simpleqa[i] for i in random.sample(range(len(simpleqa)), n)]
if mmlu is not None:
    n = min(CFG['subset_size']['mmlu'], len(mmlu))
    subsets['mmlu'] = [mmlu[i] for i in random.sample(range(len(mmlu)), n)]

print('Subsets:', {k: len(v) for k, v in subsets.items()})

In [ ]:
# Helpers (chat-template generation + log-prob picker)
def normalize_answer(s):
    return ''.join(ch.lower() for ch in str(s) if ch.isalnum() or ch.isspace()).strip()

@torch.no_grad()
def model_generate(prompt, max_new_tokens=128):
    messages = [{'role': 'user', 'content': prompt}]
    try:
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except Exception:
        text = prompt
    enc = tok(text, return_tensors='pt', truncation=True, max_length=2048).to(device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0, enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()

@torch.no_grad()
def model_pick_choice(question, choices, prompt_template='Question: {q}\nAnswer:'):
    base_prompt = prompt_template.format(q=question)
    prompt_enc = tok(base_prompt, return_tensors='pt').to(device)
    n_prompt = prompt_enc['input_ids'].shape[1]
    logprobs = []
    for choice in choices:
        full = base_prompt + ' ' + str(choice).strip()
        full_enc = tok(full, return_tensors='pt').to(device)
        n_full = full_enc['input_ids'].shape[1]
        if n_full <= n_prompt:
            logprobs.append(-1e9); continue
        logits = model(**full_enc).logits[0]
        cont_ids = full_enc['input_ids'][0, n_prompt:]
        log_probs = F.log_softmax(logits[n_prompt-1:n_full-1], dim=-1)
        lp = sum(log_probs[i, cont_ids[i]].item() for i in range(cont_ids.shape[0]))
        logprobs.append(lp / max(1, cont_ids.shape[0]))
    return int(np.argmax(logprobs)), base_prompt

In [ ]:
# Re-run benchmarks with residual capture per question
data = {bench: {'X_L31': [], 'X_L32': [], 'y': [], 'prompts': [], 'meta': []}
        for bench in subsets}

if 'truthfulqa' in subsets:
    for q in tqdm(subsets['truthfulqa'], desc='truthfulqa capture'):
        choices = q['mc1_targets']['choices']
        label = int(np.argmax(q['mc1_targets']['labels']))
        pred_idx, scoring_prompt = model_pick_choice(q['question'], choices)
        residuals = capture_last_token_residuals(scoring_prompt)
        is_hallucinated = int(pred_idx != label)
        data['truthfulqa']['X_L31'].append(residuals[31])
        data['truthfulqa']['X_L32'].append(residuals[32])
        data['truthfulqa']['y'].append(is_hallucinated)
        data['truthfulqa']['prompts'].append(scoring_prompt)
        data['truthfulqa']['meta'].append({'pred': pred_idx, 'label': label, 'q': q['question']})

if 'haluval' in subsets:
    for q in tqdm(subsets['haluval'], desc='haluval capture'):
        question = q.get('question') or q.get('input')
        right = q.get('right_answer') or q.get('answer') or ''
        prompt = f'Q: {question}\nA:'
        gen = model_generate(prompt, max_new_tokens=64)
        residuals = capture_last_token_residuals(prompt)
        is_hallucinated = int(normalize_answer(right) not in normalize_answer(gen))
        data['haluval']['X_L31'].append(residuals[31])
        data['haluval']['X_L32'].append(residuals[32])
        data['haluval']['y'].append(is_hallucinated)
        data['haluval']['prompts'].append(prompt)
        data['haluval']['meta'].append({'gen': gen[:200], 'right': str(right)[:200]})

if 'simpleqa' in subsets:
    for q in tqdm(subsets['simpleqa'], desc='simpleqa capture'):
        question = q.get('problem') or q.get('question')
        right = q.get('answer') or q.get('correct')
        prompt = f'Q: {question}\nA:'
        gen = model_generate(prompt, max_new_tokens=64)
        residuals = capture_last_token_residuals(prompt)
        is_hallucinated = int(normalize_answer(right) not in normalize_answer(gen))
        data['simpleqa']['X_L31'].append(residuals[31])
        data['simpleqa']['X_L32'].append(residuals[32])
        data['simpleqa']['y'].append(is_hallucinated)
        data['simpleqa']['prompts'].append(prompt)
        data['simpleqa']['meta'].append({'gen': gen[:200], 'right': str(right)[:200]})

if 'mmlu' in subsets:
    for q in tqdm(subsets['mmlu'], desc='mmlu capture'):
        choices = q['choices']
        label = q['answer']
        pred_idx, scoring_prompt = model_pick_choice(q['question'], choices)
        residuals = capture_last_token_residuals(scoring_prompt)
        is_hallucinated = int(pred_idx != label)
        data['mmlu']['X_L31'].append(residuals[31])
        data['mmlu']['X_L32'].append(residuals[32])
        data['mmlu']['y'].append(is_hallucinated)
        data['mmlu']['prompts'].append(scoring_prompt)
        data['mmlu']['meta'].append({'pred': pred_idx, 'label': label, 'q': q['question']})

# Convert to arrays
for bench in data:
    data[bench]['X_L31'] = np.array(data[bench]['X_L31'])
    data[bench]['X_L32'] = np.array(data[bench]['X_L32'])
    data[bench]['y']     = np.array(data[bench]['y'])
    halluc_pct = 100 * data[bench]['y'].mean()
    print(f"  {bench}: N={len(data[bench]['y'])}, hallucination rate {halluc_pct:.1f}%")

ml_hook.close()

## 4. Train linear probe — within-benchmark + cross-benchmark

**Within-bench**: 80/20 split per benchmark, train+test in same dataset. Tests if residual carries hallucination signal at all.

**Cross-bench**: train on 3 benches, evaluate on 4th. Tests if probe generalizes across hallucination types. **This is the real product test.**

In [ ]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

def train_probe(X_train, y_train, X_test, y_test, C_sweep=CFG['lr_C_sweep']):
    """L2 LR with C swept on 5-fold CV. Returns trained model + AUROC + best C."""
    if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
        return None, float('nan'), None, None
    scaler = StandardScaler().fit(X_train)
    X_tr_s = scaler.transform(X_train)
    X_te_s = scaler.transform(X_test)
    cv = StratifiedKFold(n_splits=min(5, np.bincount(y_train).min()), shuffle=True, random_state=CFG['random_seed'])
    clf = LogisticRegressionCV(
        Cs=C_sweep, cv=cv, penalty='l2', solver='lbfgs', max_iter=2000,
        scoring='roc_auc', n_jobs=-1, refit=True,
    )
    clf.fit(X_tr_s, y_train)
    y_score = clf.predict_proba(X_te_s)[:, 1]
    auroc = roc_auc_score(y_test, y_score)
    return clf, auroc, scaler, float(clf.C_[0])

# Per-benchmark train/test split (stratified by label)
from sklearn.model_selection import train_test_split
split = {}
for bench in data:
    if len(data[bench]['y']) == 0: continue
    idx_tr, idx_te = train_test_split(
        np.arange(len(data[bench]['y'])),
        test_size=1 - CFG['train_test_split'],
        stratify=data[bench]['y'] if len(np.unique(data[bench]['y'])) == 2 else None,
        random_state=CFG['random_seed'],
    )
    split[bench] = {'train_idx': idx_tr, 'test_idx': idx_te}
    print(f'  {bench}: train {len(idx_tr)}, test {len(idx_te)}')

In [ ]:
# === Within-benchmark probe AUROC ===
within_auroc = {}
for bench in data:
    if bench not in split: continue
    within_auroc[bench] = {}
    tr = split[bench]['train_idx']; te = split[bench]['test_idx']
    for layer in [31, 32]:
        Xkey = f'X_L{layer}'
        clf, auroc, scaler, C = train_probe(
            data[bench][Xkey][tr], data[bench]['y'][tr],
            data[bench][Xkey][te], data[bench]['y'][te],
        )
        within_auroc[bench][f'L{layer}'] = {
            'auroc': float(auroc), 'best_C': C,
        }
        print(f'  {bench:12s} L{layer}  AUROC = {auroc:.3f}  (C={C})')

print('\n=== Within-bench summary ===')
for bench, lr in within_auroc.items():
    best_layer = max(lr.keys(), key=lambda k: lr[k]['auroc'])
    best = lr[best_layer]
    verdict = '✅' if best['auroc'] >= 0.70 else ('🟡' if best['auroc'] >= 0.60 else '❌')
    print(f'  {verdict} {bench:12s} best layer = {best_layer}, AUROC = {best["auroc"]:.3f}')

In [ ]:
# === Cross-benchmark probe AUROC (the real product test) ===
# Train on 3 benches, test on 4th
cross_auroc = {}
all_benches = list(data.keys())
for held_out in all_benches:
    if held_out not in split: continue
    cross_auroc[held_out] = {}
    train_benches = [b for b in all_benches if b != held_out and b in split]
    for layer in [31, 32]:
        Xkey = f'X_L{layer}'
        # Concatenate train sets from all OTHER benches
        X_tr = np.concatenate([data[b][Xkey][split[b]['train_idx']] for b in train_benches])
        y_tr = np.concatenate([data[b]['y'][split[b]['train_idx']] for b in train_benches])
        # Test on held-out test split
        X_te = data[held_out][Xkey][split[held_out]['test_idx']]
        y_te = data[held_out]['y'][split[held_out]['test_idx']]
        clf, auroc, scaler, C = train_probe(X_tr, y_tr, X_te, y_te)
        cross_auroc[held_out][f'L{layer}'] = {
            'auroc': float(auroc), 'best_C': C, 'n_train': len(y_tr), 'n_test': len(y_te),
        }
        print(f'  Train on {train_benches} → eval {held_out:12s} L{layer}  AUROC = {auroc:.3f}')

print('\n=== Cross-bench summary (the GENERALIZATION test) ===')
for held_out, lr in cross_auroc.items():
    best_layer = max(lr.keys(), key=lambda k: lr[k]['auroc'])
    best = lr[best_layer]
    verdict = '✅' if best['auroc'] >= 0.65 else ('🟡' if best['auroc'] >= 0.55 else '❌')
    print(f'  {verdict} held-out {held_out:12s} best layer = {best_layer}, AUROC = {best["auroc"]:.3f}')

## 5. Compare against single SAE feature (notebook 30 baseline)

If LR probe AUROC > single feature AUROC by a meaningful margin, multi-feature linear is the right approach for production.

In [ ]:
# Pull notebook-30 numbers from HF (or hardcode current run's)
sae_feature_auroc_n30 = {
    'truthfulqa': 0.556, 'haluval': 0.500, 'simpleqa': 0.494, 'mmlu': 0.544,
}

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 5))
benches = list(within_auroc.keys())
x = np.arange(len(benches))
w = 0.27

sae_y = [sae_feature_auroc_n30.get(b, 0.5) for b in benches]
within_y = [max(within_auroc[b].values(), key=lambda v: v['auroc'])['auroc'] for b in benches]
cross_y = [max(cross_auroc[b].values(), key=lambda v: v['auroc'])['auroc'] for b in benches]

ax.bar(x - w, sae_y,    w, label='Single SAE feature (notebook 30)', color='#7f7f7f', alpha=0.85)
ax.bar(x,     within_y, w, label='LR probe within-bench (80/20)',   color='#1f77b4', alpha=0.85)
ax.bar(x + w, cross_y,  w, label='LR probe cross-bench (held-out)', color='#2ca02c', alpha=0.85)

for i, (s, w_v, c) in enumerate(zip(sae_y, within_y, cross_y)):
    ax.text(i - w, s + 0.01, f'{s:.2f}', ha='center', fontsize=9)
    ax.text(i,     w_v + 0.01, f'{w_v:.2f}', ha='center', fontsize=9, fontweight='bold')
    ax.text(i + w, c + 0.01, f'{c:.2f}', ha='center', fontsize=9, fontweight='bold')

ax.axhline(0.5, ls='--', color='gray', alpha=0.5, label='chance')
ax.axhline(0.7, ls=':',  color='red',  alpha=0.5, label='product threshold (0.70)')
ax.set_xticks(x); ax.set_xticklabels(benches)
ax.set_ylabel('AUROC (hallucination detection)')
ax.set_ylim(0.4, 1.0)
ax.set_title('HallucinationGuard v2 — single-feature vs multi-feature linear probe')
ax.legend(loc='lower right', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(LOCAL_OUT / 'auroc_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

# Tabular summary
print(f'\n{"benchmark":>14} {"SAE-single":>12} {"LR within":>11} {"LR cross":>10}')
print('-' * 50)
for b, s, w_v, c in zip(benches, sae_y, within_y, cross_y):
    print(f'{b:>14} {s:>11.3f} {w_v:>10.3f} {c:>9.3f}')

## 6. Mitigation rerun with LR probe score

If cross-bench AUROC ≥ 0.65 across benchmarks, the probe is generalizable. Test mitigation by computing best threshold and measuring confident-wrong reduction.

In [ ]:
# Pick the best-cross-bench layer per benchmark for mitigation
best_layer_per_bench = {}
for b in cross_auroc:
    best_layer_per_bench[b] = max(cross_auroc[b].keys(), key=lambda k: cross_auroc[b][k]['auroc'])
print('Best layer (cross-bench) per benchmark:', best_layer_per_bench)

# Train ONE cross-bench probe on the best-layer chosen, evaluate on each benchmark
# Use the modal best layer
from collections import Counter
modal_layer = Counter(best_layer_per_bench.values()).most_common(1)[0][0]
modal_layer_int = int(modal_layer.replace('L', ''))
Xkey = f'X_L{modal_layer_int}'
print(f'Selected probe layer: {modal_layer}')

# Probe trained on ALL benches (train splits) for production-grade signal
X_train_all = np.concatenate([data[b][Xkey][split[b]['train_idx']] for b in split])
y_train_all = np.concatenate([data[b]['y'][split[b]['train_idx']] for b in split])
scaler = StandardScaler().fit(X_train_all)
from sklearn.linear_model import LogisticRegressionCV as LRCV
global_probe = LRCV(
    Cs=CFG['lr_C_sweep'], cv=5, penalty='l2', solver='lbfgs', max_iter=2000,
    scoring='roc_auc', n_jobs=-1, refit=True,
).fit(scaler.transform(X_train_all), y_train_all)
print(f'Global probe: best C = {float(global_probe.C_[0])}')

# Score every test question under this global probe
for b in data:
    if b not in split: continue
    X_te = data[b][Xkey][split[b]['test_idx']]
    scores = global_probe.predict_proba(scaler.transform(X_te))[:, 1]
    data[b]['lr_score_test'] = scores

In [ ]:
# Threshold sweep + mitigation analysis on TEST splits
def threshold_sweep_v2(scores, labels, thresholds):
    """For (score, label=hallucinated) rows, simulate abstain mode at each threshold."""
    n = len(scores); summary = []
    for thr in thresholds:
        correct = sum(1 for s, l in zip(scores, labels) if s <= thr and l == 0)
        confident_wrong = sum(1 for s, l in zip(scores, labels) if s <= thr and l == 1)
        abstained = sum(1 for s in scores if s > thr)
        summary.append({
            'threshold': float(thr),
            'correct_pct':       100*correct/n,
            'confident_wrong_pct':   100*confident_wrong/n,
            'abstained_pct':         100*abstained/n,
            'trustworthiness': 100*(correct - confident_wrong)/n,
        })
    return summary

halluc_benches = ['truthfulqa', 'haluval', 'simpleqa']
all_test_scores = np.concatenate([data[b]['lr_score_test'] for b in halluc_benches if b in data])
all_test_labels = np.concatenate([data[b]['y'][split[b]['test_idx']] for b in halluc_benches if b in data])
thresholds = np.percentile(all_test_scores, [50, 60, 70, 75, 80, 85, 90, 95])
thresholds = sorted(set(np.round(thresholds, 3).tolist()))

sw_global = threshold_sweep_v2(all_test_scores, all_test_labels, thresholds)
best = max(sw_global, key=lambda r: r['trustworthiness'])
best_thr = best['threshold']
print(f'Best threshold (global, halluc benches): {best_thr:.3f}')
print(f'  correct={best["correct_pct"]:.1f}%  conf_wrong={best["confident_wrong_pct"]:.1f}%  '
      f'abstain={best["abstained_pct"]:.1f}%  trustworthiness={best["trustworthiness"]:+.1f}')

# Per-bench mitigation table
headline = {}
for b in data:
    if b not in split: continue
    s = data[b]['lr_score_test']
    y = data[b]['y'][split[b]['test_idx']]
    n = len(y)
    base_correct = int((y == 0).sum()); base_wrong = int((y == 1).sum())
    correct = sum(1 for ss, ll in zip(s, y) if ss <= best_thr and ll == 0)
    wrong   = sum(1 for ss, ll in zip(s, y) if ss <= best_thr and ll == 1)
    abstain = int((s > best_thr).sum())
    headline[b] = {
        'n':                       n,
        'baseline_correct_pct':    100*base_correct/n,
        'baseline_confwrong_pct':  100*base_wrong/n,
        'guard_correct_pct':       100*correct/n,
        'guard_confwrong_pct':     100*wrong/n,
        'guard_abstain_pct':       100*abstain/n,
        'confwrong_reduction_pct': 100*(base_wrong - wrong)/max(1, base_wrong),
    }

print(f'\n{"benchmark":>14} {"correct":>9} {"conf_wrong base":>16} {"conf_wrong guard":>17} {"abstained":>10} {"reduction":>10}')
print('-' * 90)
for n_b, t in headline.items():
    print(f'{n_b:>14} {t["baseline_correct_pct"]:>8.1f}% {t["baseline_confwrong_pct"]:>15.1f}% '
          f'{t["guard_confwrong_pct"]:>16.1f}% {t["guard_abstain_pct"]:>9.1f}% '
          f'{t["confwrong_reduction_pct"]:>+9.1f}%')

halluc_red = [headline[b]['confwrong_reduction_pct'] for b in halluc_benches if b in headline]
passed_mitigation = (len(halluc_red) > 0 and np.mean(halluc_red) >= 30.0)
print(f'\nMean confident-wrong reduction (halluc benches): {np.mean(halluc_red):.1f}%')
print(f'Mitigation verdict: {"✅ PASS" if passed_mitigation else "❌ FAIL"} (need ≥30%)')

# Capability check (MMLU)
if 'mmlu' in headline:
    base = headline['mmlu']['baseline_correct_pct']
    g    = headline['mmlu']['guard_correct_pct']
    delta = g - base
    passed_capability = delta >= -3.0
    print(f'\nMMLU baseline:  {base:.1f}% → guard: {g:.1f}%  (Δ {delta:+.1f}pp)')
    print(f'Capability verdict: {"✅ PASS" if passed_capability else "❌ FAIL"} (need Δ ≥ -3pp)')
else:
    passed_capability = False

## 7. Headline figure + verdict + save

In [ ]:
# Headline mitigation chart (same style as notebook 30)
fig, ax = plt.subplots(figsize=(11, 5.5))
names = []; base_wrong = []; guard_wrong = []; guard_abst = []
for b in halluc_benches:
    if b not in headline: continue
    t = headline[b]
    names.append(b); base_wrong.append(t['baseline_confwrong_pct'])
    guard_wrong.append(t['guard_confwrong_pct']); guard_abst.append(t['guard_abstain_pct'])
x = np.arange(len(names)); w = 0.4
ax.bar(x - w/2, base_wrong, w, color='#d62728', alpha=0.85, label='Baseline: confidently WRONG')
ax.bar(x + w/2, guard_wrong, w, color='#ff7f0e', alpha=0.85,
       label=f'+ HallucinationGuard v2 (LR probe, thr={best_thr:.2f}): confidently WRONG')
ax.bar(x + w/2, guard_abst, w, bottom=guard_wrong, color='#2ca02c', alpha=0.55,
       label='+ HallucinationGuard v2: ABSTAINED honestly')
for i, (b, g, a) in enumerate(zip(base_wrong, guard_wrong, guard_abst)):
    ax.text(i - w/2, b + 1, f'{b:.0f}%', ha='center', fontsize=10, color='#a01010')
    ax.text(i + w/2, g + a + 1, f'{g:.0f}%+{a:.0f}%', ha='center', fontsize=10, color='#604010')
    reduction = 100*(b-g)/max(1,b)
    ax.text(i + w/2, -3, f'-{reduction:.0f}% wrong', ha='center', fontsize=10,
             color='#0a5a0a', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=11)
ax.set_ylabel('% of questions'); ax.set_title('HallucinationGuard v2 (linear probe) — confident-wrong rate', fontsize=13)
ax.legend(loc='upper right', fontsize=10); ax.grid(axis='y', alpha=0.3)
ax.set_ylim(-7, max([b+5 for b in base_wrong] + [g+a+5 for g,a in zip(guard_wrong, guard_abst)]) + 5)
plt.tight_layout()
fig.savefig(LOCAL_OUT / 'headline.png', dpi=200, bbox_inches='tight')
fig.savefig(LOCAL_OUT / 'headline.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Final verdict
passed_detection_within = all(
    max(within_auroc[b].values(), key=lambda v: v['auroc'])['auroc'] >= 0.70
    for b in within_auroc
)
passed_detection_cross = all(
    max(cross_auroc[b].values(), key=lambda v: v['auroc'])['auroc'] >= 0.65
    for b in cross_auroc
)

verdict = {
    'within_bench_auroc':    {b: max(within_auroc[b].values(), key=lambda v: v['auroc'])['auroc']
                              for b in within_auroc},
    'cross_bench_auroc':     {b: max(cross_auroc[b].values(), key=lambda v: v['auroc'])['auroc']
                              for b in cross_auroc},
    'sae_feature_baseline':  sae_feature_auroc_n30,
    'best_threshold':        best_thr,
    'mitigation_table':      headline,
    'detection_within_pass': passed_detection_within,
    'detection_cross_pass':  passed_detection_cross,
    'mitigation_pass':       passed_mitigation,
    'capability_pass':       passed_capability,
}
n_passed = sum([passed_detection_within, passed_detection_cross, passed_mitigation, passed_capability])
verdict['summary'] = f'{n_passed}/4 thresholds passed'
if n_passed == 4:
    verdict['decision'] = '🟢 GREEN LIGHT — multi-feature LR probe is the production approach'
elif n_passed >= 2:
    verdict['decision'] = '🟡 PARTIAL — narrow product (within-bench probes) viable; universal probe needs more data/method'
else:
    verdict['decision'] = '🔴 STILL FAILS — pivot deeper (per-domain probes, ensemble, or self-consistency)'

(LOCAL_OUT / 'verdict.json').write_text(json.dumps(verdict, indent=2, default=str))

print('=' * 80)
print(f'  HallucinationGuard v2 — {verdict["summary"]}')
print('=' * 80)
print(f'  Detection within-bench (≥0.70 each):  {"✅" if passed_detection_within else "❌"}')
print(f'  Detection cross-bench  (≥0.65 each):  {"✅" if passed_detection_cross else "❌"}')
print(f'  Mitigation impact (≥30% conf_wrong reduction):  {"✅" if passed_mitigation else "❌"}')
print(f'  Capability intact (MMLU Δ ≥ -3pp):     {"✅" if passed_capability else "❌"}')
print('=' * 80)
print(f'  {verdict["decision"]}')
print('=' * 80)

# Save probe artifacts
import joblib
joblib.dump({'probe': global_probe, 'scaler': scaler, 'layer': modal_layer},
            LOCAL_OUT / 'probe.joblib')
(LOCAL_OUT / 'meta.json').write_text(json.dumps({
    'probe_layer':       modal_layer,
    'd_model':           d_model,
    'best_threshold':    float(best_thr),
    'C':                 float(global_probe.C_[0]),
    'training_benchmarks': list(split.keys()),
    'subset_sizes':      CFG['subset_size'],
}, indent=2))

# Push to HF
api = HfApi()
create_repo(CFG['hf_results_repo'], exist_ok=True, private=False, token=HF_TOKEN, repo_type='dataset')
api.upload_folder(folder_path=str(LOCAL_OUT), repo_id=CFG['hf_results_repo'], repo_type='dataset', token=HF_TOKEN)
print(f'\n✅ Artifacts pushed to https://huggingface.co/datasets/{CFG["hf_results_repo"]}')

## 8. Reading

**If 🟢 GREEN LIGHT (4/4)**: ship the **probe.joblib + scaler** as the production HallucinationGuard. Wire into `openinterp.HallucinationGuard.from_pretrained` to auto-download the probe and apply during inference. 90-day product roadmap activates.

**If 🟡 PARTIAL (2-3/4)**:
- Ship per-domain probes (Entity/Math/Knowledge variants), each with its own probe trained on within-domain data
- Bigger product surface but smaller TAM per variant
- Marketing: "OpenInterp HallucinationGuard for [Customer Support / Code / Medical]" tracks

**If 🔴 STILL FAILS**:
- The hypothesis "residual stream linearly encodes hallucination" is wrong for Qwen3.6-27B at L31/L32
- Try: deeper layers (L40+), shallow MLP probe (~3 layers), or ensemble with self-consistency (SelfCheckGPT)
- Or pivot to **EntityRecognitionGuard** vertical only (notebook 30 narrow scope) — accept smaller TAM, ship faster

## Next steps after 🟢

1. Re-instantiate `HallucinationGuard` from notebook 30 with `score_method='lr_probe'` + load `probe.joblib`
2. Re-run notebook 30's mitigation cells (no change in API)
3. Headline figure with cleaner numbers
4. Sprint S1: package probe + SDK on PyPI v0.2
5. Sprint S2: train probes for Llama-3.3, Gemma-2 (using notebook 17b/17c crosscoder transfer if Pearson_CE supports)

Both notebooks 30 + 31 become the public reproducibility artifact for OpenInterp's interpretability-native safety claim.